# Test Graph Loader

End-to-end test of `DataLoaderGraph`: mock data → universal `GraphData`.

In [1]:
import pandas as pd

from gbp.loaders import DataLoaderMock, DataLoaderGraph, GraphLoaderConfig
from gbp.graph import validate_graph, GraphDataWithQueries

## 1. Load data

In [16]:
mock = DataLoaderMock({"n": 8, "n_depots": 2, "n_timestamps": 48})

loader = DataLoaderGraph(mock, GraphLoaderConfig(distance_backend="haversine"))
loader.load_data()

2026-03-08 14:41:11 [info     ] load_start                     loader=graph
2026-03-08 14:41:11 [debug    ] source_validated               loader=graph
2026-03-08 14:41:11 [info     ] load_done                      edges=90 loader=graph nodes=10


## 2. Snapshot

In [3]:
date = loader.available_dates[12]
graph = loader.get_snapshot(date)
print(graph)

2026-03-08 14:34:26 [debug    ] snapshot                       date='2025-01-01 12:00:00' loader=graph
GraphData(nodes=10, edges=90, resources=3, commodities=1, node_attrs=1)


## 3. Inspect static parts

In [4]:
graph.nodes

,id,node_type
0,station_1,station
1,station_2,station
2,station_3,station
3,station_4,station
4,station_5,station
5,station_6,station
6,station_7,station
7,station_8,station
8,depot_1,depot
9,depot_2,depot


In [5]:
graph.coordinates

,node_id,latitude,longitude
0,station_1,40.836317,-73.919645
1,station_2,40.859020,-73.969964
2,station_3,40.828852,-73.902508
3,station_4,40.733253,-73.993022
4,station_5,40.717022,-73.975689
5,station_6,40.829500,-73.999970
6,station_7,40.756603,-73.987804
7,station_8,40.702829,-73.999381
8,depot_1,40.813905,-73.927765
9,depot_2,40.821388,-73.938729


In [6]:
graph.resources

,id,resource_type,capacity
0,truck_1,vehicle,100
1,truck_2,vehicle,100
2,truck_3,vehicle,100


In [7]:
graph.commodities

,id,commodity_type
0,bike,bike


In [8]:
graph.edges.head(10)

,source_id,target_id,distance_km,duration_h
0,station_1,station_2,4.928177,0.098564
1,station_2,station_1,4.928177,0.098564
2,station_1,station_3,1.663690,0.033274
3,station_3,station_1,1.663690,0.033274
4,station_1,station_4,13.019236,0.260385
5,station_4,station_1,13.019236,0.260385
6,station_1,station_5,14.079445,0.281589
7,station_5,station_1,14.079445,0.281589
8,station_1,station_6,6.800296,0.136006
9,station_6,station_1,6.800296,0.136006


## 4. Node attributes

In [9]:
for name, attr in graph.node_attributes.items():
    print(attr)
    display(attr.data)

AttributeTable(name='inventory_capacity', entity=node, class=capacity, granularity=['node_id'], rows=8)


,node_id,value
0,station_1,47
1,station_2,19
2,station_3,31
3,station_4,22
4,station_5,10
5,station_6,19
6,station_7,37
7,station_8,41


## 5. Inventory snapshot

In [10]:
graph.inventory

,node_id,commodity_id,quantity
0,station_1,bike,40
1,station_2,bike,17
2,station_3,bike,5
3,station_4,bike,22
4,station_5,bike,1
5,station_6,bike,9
6,station_7,bike,0
7,station_8,bike,17


## 6. Temporal comparison

In [11]:
dates = loader.available_dates[[0, 12, 24, 47]]

for d in dates:
    snap = loader.get_snapshot(d)
    total_qty = snap.inventory["quantity"].sum()
    print(f"{d}  total_quantity={total_qty}")

2026-03-08 14:34:38 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
2025-01-01 00:00:00  total_quantity=121
2026-03-08 14:34:38 [debug    ] snapshot                       date='2025-01-01 12:00:00' loader=graph
2025-01-01 12:00:00  total_quantity=111
2026-03-08 14:34:38 [debug    ] snapshot                       date='2025-01-02 00:00:00' loader=graph
2025-01-02 00:00:00  total_quantity=109
2026-03-08 14:34:38 [debug    ] snapshot                       date='2025-01-02 23:00:00' loader=graph
2025-01-02 23:00:00  total_quantity=121


## 7. Inventory time-series (raw)

In [12]:
loader.inventory_timeseries.head(10)

node_id,station_1,station_2,station_3,station_4,station_5,station_6,station_7,station_8
2025-01-01 00:00:00,43,8,16,17,10,15,4,8
2025-01-01 01:00:00,44,10,14,16,7,13,6,8
2025-01-01 02:00:00,42,12,13,16,9,13,7,10
2025-01-01 03:00:00,43,13,10,18,7,15,4,9
2025-01-01 04:00:00,40,14,7,19,5,18,2,11
2025-01-01 05:00:00,42,12,6,20,7,19,4,10
2025-01-01 06:00:00,43,15,3,22,5,19,3,12
2025-01-01 07:00:00,43,15,4,21,4,19,3,13
2025-01-01 08:00:00,43,17,7,20,2,19,1,16
2025-01-01 09:00:00,46,15,4,22,2,17,0,17


## 8. Validate graph

In [13]:
result = validate_graph(graph)
print(result)

Validation passed: no errors or warnings


## 9. Distance service

In [14]:
ids = list(graph.nodes["id"][:3])

for src in ids:
    for tgt in ids:
        if src != tgt:
            d = graph.distance_service.get_distance(src, tgt)
            print(f"{src} → {tgt}: {d:.2f} km")

station_1 → station_2: 4.93 km
station_1 → station_3: 1.66 km
station_2 → station_1: 4.93 km
station_2 → station_3: 6.59 km
station_3 → station_1: 1.66 km
station_3 → station_2: 6.59 km
